# Milestone 2


This milestone has helped to transition me from classical NLP techniques to modern, state-of-the-art deep learning architectures. It focuses on familiarizing me with the Hugging Face ecosystem, understanding how attention mechanisms create context-aware representations, and leveraging pre-trained models and zero-shot classification to drastically improve upon my baseline ranking metrics.


Topic Readings:

- Hugging Face transformers & datasets Library Basics  

- The Attention Mechanism and BERT/RoBERTa Architectures

- Dense Context-Aware Embeddings (e.g., Sentence-Transformers) 

- Zero-Shot Classification

- Softmax vs. Independent Sigmoid (Multi-label) Probabilities  

- Prompting Small Language Models (SLMs) for Generative QA

---
---

### Question 1:


Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.




In [1]:
from datasets import load_dataset

train_ds = load_dataset('csv', data_files='../data/train.csv')['train']

def combine(d):
    d['combined_text'] = d['prompt'] + ' ' + d['A']
    return d
train_map = train_ds.map(combine)

len(train_map[51]['combined_text'])

/home/deeepak/iitm_project/smart-mcq-solver-dlgenai-2026/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


614

### Question 2:


Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.vocab_size)

30522


### Question 3:

Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [3]:
tokenizer.sep_token_id

102

### Question 4:

Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [4]:
encoded = tokenizer(
    train_ds[:]['prompt'],
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

encoded.input_ids.shape

torch.Size([2000, 128])

### Question 5:

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [5]:
head_dim = 768 // 12
head_dim

64

### Question 6:

Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [6]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(train_ds[0]['prompt'], return_tensors='pt')
outputs = model(**inputs)
outputs.last_hidden_state.shape

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2487.59it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

### Question 7:

 Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [7]:
cls_vec = outputs.last_hidden_state[0, 0]
sum_first5 = cls_vec[:5].sum().item()
round(sum_first5, 4)

-1.2001

### Question 8:

Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [8]:
model_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

text = "Light-ion fusion is a technique."
inputs_fusion = tokenizer(text, return_tensors="pt")
outputs_attn = model_attn(**inputs_fusion)

tokens = tokenizer.convert_ids_to_tokens(inputs_fusion.input_ids[0])
fusion_index = tokens.index("fusion")

attention_weight = outputs_attn.attentions[-1][0, 0, 0, fusion_index].item()
round(attention_weight, 4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5469.60it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.1025

### Question 9


Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.



In [9]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_prompt = st_model.encode(train_ds[0]['prompt'], convert_to_tensor=True)
emb_B = st_model.encode(train_ds[0]['B'], convert_to_tensor=True)
sim_score = util.cos_sim(emb_prompt, emb_B).item()
print(round(sim_score, 4))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4190.52it/s]


0.7658


### Question 10:

Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [10]:
def map3(true_answer, ranked_answers):
    if true_answer in ranked_answers[:3]:
        return 1.0 / (ranked_answers[:3].index(true_answer) + 1)
    return 0.0

In [11]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(train['prompt'])

tfidf_top3 = []
for _, row in train.iterrows():
    prompt_vec = vectorizer.transform([row['prompt']])
    option_scores = [
        ('A', cosine_similarity(prompt_vec, vectorizer.transform([row['A']]))[0,0]),
        ('B', cosine_similarity(prompt_vec, vectorizer.transform([row['B']]))[0,0]),
        ('C', cosine_similarity(prompt_vec, vectorizer.transform([row['C']]))[0,0]),
        ('D', cosine_similarity(prompt_vec, vectorizer.transform([row['D']]))[0,0]),
        ('E', cosine_similarity(prompt_vec, vectorizer.transform([row['E']]))[0,0]),
    ]
    ranked_answers = [opt for opt, _ in sorted(option_scores,key=lambda x:x[1],reverse=True)]
    tfidf_top3.append(ranked_answers[:3])




In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

minilm_top3 = []
for _, row in train.iterrows():
    prompt_embedding = model.encode(row["prompt"], normalize_embeddings=True)
    option_scores = []
    for option in ["A", "B", "C", "D", "E"]:
        option_embedding = model.encode(row[option],normalize_embeddings=True)
        score = cosine_similarity([prompt_embedding],[option_embedding])[0,0]
        option_scores.append((option, score))

    ranked_answers = [opt for opt, _ in sorted(option_scores,key=lambda x: x[1],reverse=True)]
    minilm_top3.append(ranked_answers[:3])


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8956.99it/s]


In [13]:
count = 0

for i, row in train.iterrows():

    actual = row["answer"]

    if actual not in tfidf_top3[i] and actual in minilm_top3[i]:
        count += 1

print(count)

315


### Question 11:

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [14]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", device="cuda")

row = train.iloc[1]
result = classifier(
    row["prompt"],
    candidate_labels=[row["A"], row["B"], row["C"]],
)

top_prob = round(float(result["scores"][0]), 4)
print(top_prob)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1836.52it/s]


0.4575


### Question 12:

Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [15]:
result_multilabel = classifier(
    row["prompt"],
    candidate_labels=[row["A"], row["B"], row["C"]],
    multi_label=True,
)

diff = abs(sum(result["scores"]) - sum(result_multilabel["scores"]))
round(diff, 4)

0.9995

### Question 13:

Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [ ]:
# flan_pipe = pipeline("text-generation", model="google/flan-t5-small")

# row0 = train.iloc[0]
# prompt_text = (
#     f"Question: {row0['prompt']}. "
#     f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
#     "Answer with just the letter A or B."
# )

# output = flan_pipe(prompt_text, max_new_tokens=5)
# print(output[0]["generated_text"])

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6783.84it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadMo

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row0 = train.iloc[0]
prompt_text = (
    f"Question: {row0['prompt']}. "
    f"Is the correct answer A: {row0['A']} or B: {row0['B']}? "
    "Answer with just the letter A or B."
)

inputs = tokenizer(prompt_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(answer)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6933.52it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


B
